# Dicionário de Dados · Conecta Saúde

Este notebook documenta **todos os arquivos gerados** pelos notebooks de extração, em `dados/tratado/`.

Ele não é um documento escrito à mão. É um **gerador**: lê os Parquet que existem no disco, extrai o schema real, calcula estatísticas de preenchimento e cruza tudo com um catálogo curado de descrições. A consequência prática é que o dicionário nunca descreve uma coluna que não existe mais, nem deixa de listar uma coluna nova — ele reclama quando encontra coluna sem descrição.

---

## O que ele produz

Três saídas em `dados/tratado/_dicionario/`:

| Arquivo | Para quê |
|---|---|
| `DICIONARIO_DE_DADOS.md` | Leitura humana, pronto para anexar à entrega |
| `dicionario_dados.csv` | Legível por máquina, uma linha por coluna |
| `dicionario_tabelas.csv` | Uma linha por arquivo, com contagens e chaves |

## Por que gerar em vez de escrever

Um dicionário digitado à mão começa correto e envelhece em silêncio. Quando alguém acrescenta uma coluna na extração e esquece de atualizar o documento, o dicionário passa a mentir — e ninguém percebe, porque nada quebra.

Aqui, cada coluna encontrada no disco precisa ter entrada no catálogo. Se não tiver, ela aparece na lista de pendências ao final da execução. O documento pode ficar incompleto, mas não fica errado sem avisar.

In [ ]:
# Só pandas e pyarrow.
%pip install -q pandas pyarrow

## 1. Parâmetros e descoberta dos arquivos

Os arquivos são agrupados em **famílias**: os 27 `cnes_lt_AC_2024`, `cnes_lt_AL_2024`… compartilham o mesmo schema e viram uma entrada só, `cnes_lt_{UF}_{ANO}`. Documentar 27 vezes a mesma estrutura não ajudaria ninguém.

In [ ]:
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd


def raiz_do_projeto() -> Path:
    '''Devolve a pasta do repositório, subindo até encontrar o .git.'''
    atual = Path.cwd().resolve()
    for pasta in (atual, *atual.parents):
        if (pasta / ".git").exists():
            return pasta
    return atual


RAIZ = raiz_do_projeto()
DIRETORIO_TRATADO = RAIZ / "dados" / "tratado"
DIRETORIO_DICIONARIO = DIRETORIO_TRATADO / "_dicionario"
DIRETORIO_DICIONARIO.mkdir(parents=True, exist_ok=True)

# Colunas de baixa cardinalidade têm o domínio observado listado no dicionário.
LIMITE_DOMINIO = 25


def familia_do_arquivo(caminho: Path) -> str:
    '''Reduz o nome a um padrão, para que 27 arquivos de UF virem uma entrada.'''
    base = caminho.stem
    base = re.sub(r"_[A-Z]{2}_(?=\d)", "_{UF}_", base)   # _SP_2024 -> _{UF}_2024
    base = re.sub(r"\d{6}$", "{COMPET}", base)           # _202412  -> _{COMPET}
    base = re.sub(r"\d{4}$", "{ANO}", base)              # _2024    -> _{ANO}
    return base


arquivos = sorted(DIRETORIO_TRATADO.glob("**/*.parquet"))
familias = defaultdict(list)
for caminho in arquivos:
    if caminho.parent.name == "_dicionario":
        continue
    familias[(caminho.parent.name, familia_do_arquivo(caminho))].append(caminho)

print(f"{len(arquivos)} arquivos Parquet em {len(familias)} famílias\n")
for (fonte, base), lista in sorted(familias.items()):
    print(f"  {fonte:<8} {base:<52} {len(lista):>3} arquivo(s)")

## 2. O catálogo de descrições

Este é o conteúdo escrito à mão — a parte que nenhum script consegue inferir. Cada entrada diz o que a coluna significa, de onde veio e, quando existe, qual armadilha ela esconde.

As chaves são nomes de coluna. Colunas com o mesmo nome em arquivos diferentes compartilham a descrição, o que é proposital: `CNES` significa a mesma coisa no CNES e no SIH, e é exatamente por isso que a junção entre eles funciona.

In [ ]:
# (descrição, origem, observação)
CATALOGO = {
    # ---------------------------------------------------------------- chaves
    "CNES": (
        "Código do estabelecimento de saúde no Cadastro Nacional",
        "CNES / SIH",
        "7 dígitos, com zero à esquerda. É a chave que liga leitos a internações. "
        "Se virar inteiro, o zero some e a junção falha em silêncio.",
    ),
    "CODUFMUN": (
        "Código IBGE do município do estabelecimento, sem dígito verificador",
        "CNES",
        "6 dígitos. Corresponde aos 6 primeiros do código IBGE de 7.",
    ),
    "UF": ("Sigla da unidade federativa", "atribuída na extração", ""),
    "COMPETENCIA": (
        "Competência de referência no formato AAAAMM",
        "atribuída na extração a partir do nome do arquivo",
        "",
    ),
    "COMPETEN": (
        "Competência declarada dentro do próprio arquivo do DATASUS",
        "CNES",
        "Redundante com COMPETENCIA; mantida para conferência de integridade.",
    ),
    "GRUPO_CNES": ("Grupo do arquivo CNES: LT (leitos) ou ST (estabelecimentos)", "extração", ""),

    # ------------------------------------------------------------- CNES leitos
    "TP_LEITO": (
        "Tipo de leito",
        "CNES grupo LT",
        "Domínio oficial em CNV/tip1leit.cnv. O tipo 7 (Hospital/DIA) é excluído "
        "da contagem do índice por não ser leito de permanência.",
    ),
    "CODLEITO": (
        "Especialidade do leito",
        "CNES grupo LT",
        "Domínio em CNV/Esp_leit.CNV. Os códigos são disjuntos por TP_LEITO, "
        "o que garante que não há dupla contagem entre tipos.",
    ),
    "TIPO_LEITO_DESC": ("Descrição do tipo de leito", "CNV/tip1leit.cnv do TAB_CNES.zip", ""),
    "ESPECIALIDADE_DESC": (
        "Descrição da especialidade do leito",
        "CNV/Esp_leit.CNV do TAB_CNES.zip",
        "O código 96 aparece nos dados e não existe no CNV; recebe o rótulo "
        "'(não catalogado no CNV)'.",
    ),
    "QT_EXIST": ("Leitos existentes no estabelecimento", "CNES grupo LT", "Total, SUS e não-SUS somados."),
    "QT_CONTR": ("Leitos contratados", "CNES grupo LT", ""),
    "QT_SUS": (
        "Leitos disponíveis ao SUS",
        "CNES grupo LT",
        "Subconjunto de QT_EXIST, não uma medida independente. É o denominador do ICPA.",
    ),
    "QT_NSUS": (
        "Leitos não disponíveis ao SUS",
        "CNES grupo LT",
        "Igual a QT_EXIST menos QT_SUS, conferido linha a linha na extração.",
    ),
    "leitos_sus": ("Soma de leitos SUS no grão da tabela", "agregação de QT_SUS", ""),
    "leitos_existentes": ("Soma de leitos existentes no grão da tabela", "agregação de QT_EXIST", ""),
    "estabelecimentos_com_leito": (
        "Estabelecimentos distintos com ao menos um leito SUS",
        "contagem distinta de CNES",
        "Não é somável entre municípios: um estabelecimento não se divide.",
    ),

    # ------------------------------------------------ CNES estabelecimentos
    # As colunas *_DESC vêm dos CNV do TAB_CNES.zip, declarados no
    # Estabelecimento.def. Sem elas o ST é uma tabela de códigos opacos.
    "TP_UNID_DESC": ("Descrição do tipo de estabelecimento", "CNV/TP_ESTAB.CNV", ""),
    "TPGESTAO_DESC": ("Descrição do tipo de gestão", "CNV/TPGESTAO.CNV", ""),
    "NAT_JUR_DESC": (
        "Descrição da natureza jurídica",
        "CNV/NATJURC.CNV, com recuo para o grupo do 1º dígito",
        "Códigos de grupo como 4000 (Pessoa Física) não constam do CNV específico e "
        "recebem o rótulo do grupo. Sem esse recuo, 26% dos estabelecimentos ficariam sem nome.",
    ),
    "ATIVIDAD_DESC": (
        "Descrição da atividade de ensino e pesquisa",
        "CNV/Ativ_Ens.CNV",
        "Separa hospital de ensino, que concentra 52 mil leitos SUS.",
    ),
    "NIV_DEP_DESC": ("Descrição do nível de dependência", "CNV/NIVELDEP.CNV", ""),
    "TURNO_AT_DESC": ("Descrição do turno de atendimento", "CNV/TurnosAt.cnv", ""),
    "VINC_SUS_DESC": ("Descrição do vínculo com o SUS", "CNV/Vinc_SUS.cnv", ""),
    "CLIENTEL_DESC": ("Descrição do fluxo de clientela", "CNV/Flux_Cli.CNV", ""),
    "TP_UNID": ("Tipo de estabelecimento", "CNES grupo ST", "Hospital geral, UBS, pronto-socorro etc."),
    "TPGESTAO": (
        "Tipo de gestão do estabelecimento",
        "CNES grupo ST",
        "M = municipal, E = estadual, D = dupla. Não confundir com esfera administrativa.",
    ),
    "NAT_JUR": ("Natureza jurídica", "CNES grupo ST", "Código da tabela de naturezas jurídicas do IBGE/RFB."),
    "ATIVIDAD": ("Atividade de ensino e pesquisa", "CNES grupo ST", ""),
    "NIV_DEP": ("Nível de dependência: mantida ou individual", "CNES grupo ST", ""),
    "TURNO_AT": ("Turno de atendimento", "CNES grupo ST", ""),
    "VINC_SUS": ("Vínculo com o SUS", "CNES grupo ST", ""),
    "CLIENTEL": ("Fluxo de clientela atendida", "CNES grupo ST", ""),

    # ------------------------------------------------------------------ IBGE
    "cod_municipio_ibge": (
        "Código IBGE do município, com dígito verificador",
        "IBGE / API de Localidades",
        "7 dígitos. É o padrão oficial do IBGE.",
    ),
    "cod_municipio_datasus": (
        "Código do município no padrão DATASUS",
        "derivado: 6 primeiros dígitos do código IBGE",
        "É a chave de junção com CODUFMUN, MUNIC_RES, MUNIC_MOV e PA_UFMUN. "
        "O corte foi conferido: não gera colisão entre municípios.",
    ),
    "municipio": ("Nome do município", "IBGE", ""),
    "cod_uf": ("Código IBGE da unidade federativa", "IBGE", ""),
    "uf": ("Sigla da unidade federativa", "IBGE", ""),
    "estado": ("Nome da unidade federativa", "IBGE", ""),
    "cod_regiao": ("Código da região", "IBGE", ""),
    "regiao": ("Região do país", "IBGE", "Norte, Nordeste, Sudeste, Sul, Centro-Oeste."),
    "cod_microrregiao": ("Código da microrregião", "IBGE", ""),
    "microrregiao": ("Nome da microrregião", "IBGE", "Recorte histórico, substituído pelas regiões imediatas."),
    "cod_mesorregiao": ("Código da mesorregião", "IBGE", ""),
    "mesorregiao": ("Nome da mesorregião", "IBGE", "Recorte histórico."),
    "cod_regiao_imediata": ("Código da região geográfica imediata", "IBGE", ""),
    "regiao_imediata": ("Nome da região geográfica imediata", "IBGE", "Recorte territorial vigente."),
    "cod_regiao_intermediaria": ("Código da região geográfica intermediária", "IBGE", ""),
    "regiao_intermediaria": ("Nome da região geográfica intermediária", "IBGE", "Recorte territorial vigente."),
    "populacao": (
        "População residente estimada",
        "IBGE / SIDRA tabela 6579, variável 9324",
        "Denominador de todo indicador por habitante do projeto.",
    ),
    "ano_populacao": ("Ano de referência da estimativa populacional", "IBGE", ""),

    # ---------------------------------------------------------------- SIGTAP
    "CO_PROCEDIMENTO": (
        "Código do procedimento na Tabela Unificada",
        "SIGTAP / tb_procedimento",
        "10 dígitos, todos com zero à esquerda. Casa com PROC_REA do SIH e "
        "PA_PROC_ID do SIA. A hierarquia está dentro do código: "
        "grupo (2) + subgrupo (2) + forma de organização (2) + sequencial (3) + DV (1).",
    ),
    "NO_PROCEDIMENTO": ("Nome do procedimento", "SIGTAP", ""),
    "CO_GRUPO": ("Código do grupo do procedimento", "SIGTAP", "Primeiros 2 dígitos do código."),
    "NO_GRUPO": ("Nome do grupo do procedimento", "SIGTAP / tb_grupo", ""),
    "CO_SUB_GRUPO": ("Código do subgrupo", "SIGTAP", "Dígitos 3 e 4 do código."),
    "NO_SUB_GRUPO": ("Nome do subgrupo", "SIGTAP / tb_sub_grupo", ""),
    "CO_FORMA_ORGANIZACAO": ("Código da forma de organização", "SIGTAP", "Dígitos 5 e 6 do código."),
    "NO_FORMA_ORGANIZACAO": ("Nome da forma de organização", "SIGTAP / tb_forma_organizacao", ""),
    "TP_COMPLEXIDADE": (
        "Código da complexidade do procedimento",
        "SIGTAP",
        "0 = não se aplica, 1 = atenção básica, 2 = média, 3 = alta.",
    ),
    "NO_COMPLEXIDADE": ("Descrição da complexidade", "derivado de TP_COMPLEXIDADE", ""),
    "CO_FINANCIAMENTO": ("Código da fonte de financiamento", "SIGTAP", ""),
    "NO_FINANCIAMENTO": ("Nome da fonte de financiamento", "SIGTAP / tb_financiamento", "PAB, MAC, FAEC etc."),
    "VL_HOSPITALAR": ("Valor hospitalar do procedimento, em reais", "SIGTAP", "Convertido de centavos na extração."),
    "VL_AMBULATORIAL": ("Valor ambulatorial do procedimento, em reais", "SIGTAP", "Convertido de centavos."),
    "VL_PROFISSIONAL": ("Valor profissional do procedimento, em reais", "SIGTAP", "Convertido de centavos."),
    "VL_TOTAL": ("Soma dos três valores do procedimento, em reais", "derivado", "Valor de tabela, não valor pago."),
    "QT_DIAS_PERMANENCIA": ("Dias de permanência previstos em tabela", "SIGTAP", "Previsão normativa, não realizado."),
    "QT_PONTOS": ("Pontuação do procedimento", "SIGTAP", ""),
    "DT_COMPETENCIA": ("Competência da tabela SIGTAP", "SIGTAP", "A tabela muda todo mês."),

    # ------------------------------------------------------------------- SIH
    "MUNIC_RES": (
        "Código do município de residência do paciente",
        "SIH grupo RD",
        "6 dígitos. Cruzado com a população do IBGE, mede NECESSIDADE da população.",
    ),
    "MUNIC_MOV": (
        "Código do município onde a internação ocorreu",
        "SIH grupo RD",
        "6 dígitos. Cruzado com os leitos do CNES, mede CARGA sobre a estrutura. "
        "Trocar por MUNIC_RES inverte a leitura nos municípios-polo.",
    ),
    "PROC_REA": (
        "Código do procedimento realizado na internação",
        "SIH grupo RD",
        "10 dígitos com zero à esquerda; casa com CO_PROCEDIMENTO do SIGTAP.",
    ),
    "internacoes": ("Contagem de AIH no grão da tabela", "SIH grupo RD", "Uma AIH é uma internação paga pelo SUS."),
    "obitos": ("Internações com desfecho de óbito", "SIH, campo MORTE", ""),
    "internacoes_com_uti": ("Internações que usaram ao menos uma diária de UTI", "SIH, campo UTI_MES_TO", ""),
    "internacoes_urgencia": (
        "Internações de caráter de urgência",
        "SIH, campo CAR_INT",
        "Complementar às eletivas.",
    ),
    "dias_permanencia": ("Soma dos dias de permanência", "SIH, campo DIAS_PERM", ""),
    "diarias_uti": ("Soma das diárias de UTI", "SIH, campo UTI_MES_TO", ""),
    "valor_total": ("Soma do valor das AIH, em reais", "SIH, campo VAL_TOT", "Valor efetivamente pago."),
    "permanencia_media": ("Dias de permanência divididos pelo número de internações", "derivado", ""),

    # ------------------------------------------------------------------- SIA
    "PA_PROC_ID": (
        "Código do procedimento ambulatorial",
        "SIA grupo PA",
        "10 dígitos; casa com CO_PROCEDIMENTO do SIGTAP.",
    ),
    "PA_UFMUN": ("Código do município do estabelecimento executante", "SIA grupo PA", "6 dígitos."),
    "PA_MUNPCN": ("Código do município de residência do paciente", "SIA grupo PA", "6 dígitos."),
    "PA_NIVCPL": (
        "Nível de complexidade do procedimento",
        "SIA grupo PA",
        "0 = não se aplica, 1 = atenção básica, 2 = média, 3 = alta.",
    ),
    "quantidade_aprovada": (
        "Quantidade de procedimentos aprovada na auditoria",
        "SIA, campo PA_QTDAPR",
        "É a medida de produção efetiva. Use esta, não a apresentada.",
    ),
    "quantidade_apresentada": (
        "Quantidade de procedimentos apresentada pelo estabelecimento",
        "SIA, campo PA_QTDPRO",
        "A diferença para a aprovada é a produção glosada.",
    ),
    "valor_aprovado": ("Valor aprovado, em reais", "SIA, campo PA_VALAPR", ""),
    "quantidade_outro_municipio": (
        "Produção realizada para pacientes de outro município",
        "SIA, campo PA_MNDIF",
        "Medida de invasão: quanto da produção local atende gente de fora.",
    ),
    "registros": ("Número de linhas do arquivo original agregadas nesta linha", "contagem", "Não é volume de atendimento."),

    # ---------------------------------------------------------------- SISREG
    "cod_municipio_solicitante": ("Código do município que solicitou a vaga", "regulação", "6 dígitos."),
    "especialidade": ("Especialidade ou procedimento solicitado", "regulação", ""),
    "solicitacoes": ("Total de solicitações", "regulação", ""),
    "na_fila": (
        "Solicitações ainda não executadas",
        "regulação",
        "Inclui quem espera há muito tempo — contar só as executadas esconderia a fila pior.",
    ),
    "executadas": ("Solicitações já realizadas", "regulação", ""),
    "espera_mediana": ("Mediana de dias de espera", "regulação", "Para quem ainda espera, conta-se até a data de corte."),
    "espera_media": ("Média de dias de espera", "regulação", "Sensível à cauda longa; prefira a mediana."),
    "fora_do_municipio": ("Solicitações encaminhadas para outro município", "regulação", ""),
    "taxa_evasao_pct": ("Percentual de solicitações encaminhadas para fora", "derivado", ""),
    "ORIGEM_DADO": (
        "Procedência do dado: REAL ou SINTETICO",
        "controle do pipeline",
        "SINTETICO indica base simulada para desenvolvimento. "
        "Números com essa marca NÃO representam o SUS e não podem ser apresentados como tal.",
    ),
}

print(f"{len(CATALOGO)} colunas catalogadas")

## 3. Perfilamento dos arquivos

Para cada família, o notebook mede o que está lá: tipo, preenchimento, cardinalidade, faixa de valores e exemplos. Colunas com poucos valores distintos têm o domínio observado listado por extenso — é o que transforma `TPGESTAO` de sigla opaca em informação utilizável.

**O perfil cobre todos os arquivos da família, não uma amostra.** Isso importa: perfilando só `cnes_lt_AC_2024`, a coluna `UF` apareceria com um único valor distinto e `CODUFMUN` com dezoito, o que descreveria o Acre e não o Brasil. Um dicionário que erra a cardinalidade é pior que nenhum, porque parece confiável.

O custo é baixo porque a camada tratada inteira tem poucos megabytes — todos os arquivos já são agregados. Se algum dia uma família crescer a ponto de não caber, é aqui que o perfilamento precisa virar amostragem declarada.

In [ ]:
def perfilar_coluna(serie: pd.Series) -> dict:
    '''Mede tipo, preenchimento, cardinalidade e faixa de uma coluna.'''
    total = len(serie)
    nulos = int(serie.isna().sum())
    distintos = int(serie.nunique(dropna=True))

    perfil = {
        "tipo": str(serie.dtype),
        "nulos": nulos,
        "pct_nulo": round(100 * nulos / total, 2) if total else 0.0,
        "distintos": distintos,
        "dominio_observado": "",
        "minimo": "",
        "maximo": "",
        "exemplos": "",
    }

    validos = serie.dropna()
    if validos.empty:
        return perfil

    if pd.api.types.is_numeric_dtype(serie):
        perfil["minimo"] = f"{validos.min():,.2f}".rstrip("0").rstrip(".")
        perfil["maximo"] = f"{validos.max():,.2f}".rstrip("0").rstrip(".")
    else:
        # Domínio por extenso só quando é curto o bastante para ser útil.
        if distintos <= LIMITE_DOMINIO:
            perfil["dominio_observado"] = " | ".join(sorted(validos.unique().astype(str))[:LIMITE_DOMINIO])
        perfil["minimo"] = str(validos.min())
        perfil["maximo"] = str(validos.max())

    perfil["exemplos"] = ", ".join(validos.astype(str).head(3).tolist())
    return perfil


linhas_colunas, linhas_tabelas = [], []

for (fonte, base), caminhos in sorted(familias.items()):
    representativo = caminhos[0]

    # A família inteira, para que cardinalidade e domínio descrevam o conjunto
    # real e não o primeiro arquivo por acaso.
    df = (
        pd.read_parquet(representativo)
        if len(caminhos) == 1
        else pd.concat([pd.read_parquet(c) for c in caminhos], ignore_index=True)
    )

    total_linhas = len(df)
    tamanho_mb = sum(c.stat().st_size for c in caminhos) / 1024**2

    linhas_tabelas.append({
        "fonte": fonte,
        "tabela": base,
        "arquivos": len(caminhos),
        "linhas": total_linhas,
        "colunas": df.shape[1],
        "tamanho_mb": round(tamanho_mb, 3),
        "exemplo_arquivo": representativo.name,
        "perfil_sobre": "família completa" if len(caminhos) > 1 else "arquivo único",
    })

    for coluna in df.columns:
        descricao, origem, observacao = CATALOGO.get(coluna, ("", "", ""))
        linhas_colunas.append({
            "fonte": fonte,
            "tabela": base,
            "coluna": coluna,
            "descricao": descricao,
            "origem": origem,
            "observacao": observacao,
            **perfilar_coluna(df[coluna]),
        })

dicionario = pd.DataFrame(linhas_colunas)
tabelas = pd.DataFrame(linhas_tabelas)

print(f"{len(tabelas)} tabelas, {len(dicionario)} colunas documentadas")
print(f"{tabelas['linhas'].sum():,} linhas no total, {tabelas['tamanho_mb'].sum():.1f} MB\n")
display(tabelas)

## 4. Pendências do catálogo

A checagem que mantém o dicionário honesto: toda coluna presente no disco precisa ter descrição. As que não têm aparecem aqui — o documento sai mesmo assim, mas com a lacuna visível em vez de escondida.

In [ ]:
sem_descricao = dicionario[dicionario["descricao"] == ""]

if sem_descricao.empty:
    print("Todas as colunas têm descrição no catálogo.")
else:
    print(f"{sem_descricao['coluna'].nunique()} coluna(s) sem descrição — acrescente ao CATALOGO:\n")
    for coluna, grupo in sem_descricao.groupby("coluna"):
        tabelas_txt = ", ".join(sorted(grupo["tabela"].unique()))
        print(f'    "{coluna}": ("", "", ""),    # em {tabelas_txt}')

cobertura = 100 * (1 - len(sem_descricao) / len(dicionario))
print(f"\nCobertura do catálogo: {cobertura:.1f}%")

## 5. Geração do documento

O Markdown é montado a partir do mesmo DataFrame que vira CSV — não há uma segunda fonte de verdade que possa divergir.

In [ ]:
def gerar_markdown(tabelas: pd.DataFrame, dicionario: pd.DataFrame) -> str:
    '''Monta o documento a partir dos DataFrames perfilados.'''
    from datetime import date

    partes = [
        "# Dicionário de Dados — Conecta Saúde",
        "",
        f"Gerado automaticamente em {date.today().isoformat()} a partir dos arquivos "
        f"em `dados/tratado/`.",
        "",
        "Não edite este arquivo à mão: ele é sobrescrito a cada execução do notebook "
        "`dicionario_de_dados_conecta_saude.ipynb`. Para alterar uma descrição, "
        "edite o `CATALOGO` na seção 2 do notebook.",
        "",
        "## Visão geral",
        "",
        "| Fonte | Tabela | Arquivos | Linhas | Colunas | MB |",
        "|---|---|---:|---:|---:|---:|",
    ]
    for t in tabelas.itertuples():
        partes.append(
            f"| {t.fonte} | `{t.tabela}` | {t.arquivos} | {t.linhas:,} | "
            f"{t.colunas} | {t.tamanho_mb:.2f} |"
        )

    partes += ["", f"**Total:** {tabelas['linhas'].sum():,} linhas em "
               f"{tabelas['arquivos'].sum()} arquivos, {tabelas['tamanho_mb'].sum():.1f} MB.", ""]

    for fonte in sorted(tabelas["fonte"].unique()):
        partes += ["", f"---", "", f"# Fonte: {fonte.upper()}", ""]

        for t in tabelas[tabelas["fonte"] == fonte].itertuples():
            partes += [
                f"## `{t.tabela}`",
                "",
                f"{t.linhas:,} linhas · {t.colunas} colunas · {t.arquivos} arquivo(s) · "
                f"{t.tamanho_mb:.2f} MB",
                "",
                f"Exemplo de arquivo: `{t.exemplo_arquivo}`",
                "",
                "| Coluna | Tipo | Descrição | Nulos | Distintos | Exemplos |",
                "|---|---|---|---:|---:|---|",
            ]
            colunas = dicionario[
                (dicionario["fonte"] == fonte) & (dicionario["tabela"] == t.tabela)
            ]
            for c in colunas.itertuples():
                descricao = c.descricao or "_(sem descrição no catálogo)_"
                exemplos = str(c.exemplos)[:48]
                partes.append(
                    f"| `{c.coluna}` | {c.tipo} | {descricao} | "
                    f"{c.pct_nulo:.1f}% | {c.distintos:,} | {exemplos} |"
                )
            partes.append("")

            # Detalhamento só do que tem conteúdo extra a dizer.
            detalhes = colunas[(colunas["observacao"] != "") | (colunas["dominio_observado"] != "")]
            if not detalhes.empty:
                partes += ["<details>", "<summary>Domínios e observações</summary>", ""]
                for c in detalhes.itertuples():
                    partes.append(f"**`{c.coluna}`**")
                    if c.origem:
                        partes.append(f"- Origem: {c.origem}")
                    if c.dominio_observado:
                        partes.append(f"- Valores observados: `{c.dominio_observado}`")
                    if c.observacao:
                        partes.append(f"- {c.observacao}")
                    partes.append("")
                partes += ["</details>", ""]

    return "\n".join(partes)


markdown = gerar_markdown(tabelas, dicionario)

destino_md = DIRETORIO_DICIONARIO / "DICIONARIO_DE_DADOS.md"
destino_md.write_text(markdown, encoding="utf-8")

destino_colunas = DIRETORIO_DICIONARIO / "dicionario_dados.csv"
dicionario.to_csv(destino_colunas, index=False, encoding="utf-8-sig")

destino_tabelas = DIRETORIO_DICIONARIO / "dicionario_tabelas.csv"
tabelas.to_csv(destino_tabelas, index=False, encoding="utf-8-sig")

dicionario.to_parquet(DIRETORIO_DICIONARIO / "dicionario_dados.parquet", index=False)

print(f"Pasta: {DIRETORIO_DICIONARIO}\n")
for arq in sorted(DIRETORIO_DICIONARIO.iterdir()):
    print(f"  {arq.name:<32} {arq.stat().st_size / 1024:>8.1f} KB")

print(f"\nO Markdown tem {len(markdown.splitlines()):,} linhas.")

## 6. Conferência das chaves de junção

Um dicionário que só descreve colunas isoladas não conta a parte mais importante: como as tabelas se ligam. A célula abaixo verifica, com os dados reais, se as chaves declaradas de fato casam.

In [ ]:
JUNCOES = [
    ("ibge", "cod_municipio_datasus", "cnes", "CODUFMUN", "população por município com leito"),
    ("ibge", "cod_municipio_datasus", "sih", "MUNIC_RES", "internações por 10 mil habitantes"),
    ("ibge", "cod_municipio_datasus", "sia", "PA_UFMUN", "produção ambulatorial por habitante"),
    ("sigtap", "CO_PROCEDIMENTO", "sih", "PROC_REA", "nome do procedimento internado"),
    ("sigtap", "CO_PROCEDIMENTO", "sia", "PA_PROC_ID", "nome do procedimento ambulatorial"),
    ("cnes", "CNES", "sih", "CNES", "leitos e internações do mesmo hospital"),
]


def valores_da_coluna(fonte: str, coluna: str) -> set:
    '''Junta os valores distintos de uma coluna em todos os arquivos da fonte.'''
    encontrados = set()
    for caminho in (DIRETORIO_TRATADO / fonte).glob("*.parquet"):
        try:
            serie = pd.read_parquet(caminho, columns=[coluna])[coluna]
        except (ValueError, KeyError):
            continue
        encontrados.update(serie.dropna().astype(str).unique())
    return encontrados


resultados = []
for fonte_a, col_a, fonte_b, col_b, proposito in JUNCOES:
    a = valores_da_coluna(fonte_a, col_a)
    b = valores_da_coluna(fonte_b, col_b)
    if not a or not b:
        resultados.append({
            "de": f"{fonte_a}.{col_a}", "para": f"{fonte_b}.{col_b}",
            "para_que": proposito, "situacao": "fonte ainda não extraída",
            "casam": "", "orfaos": "",
        })
        continue
    orfaos = b - a
    resultados.append({
        "de": f"{fonte_a}.{col_a}", "para": f"{fonte_b}.{col_b}",
        "para_que": proposito,
        "situacao": "OK" if not orfaos else "há órfãos",
        "casam": f"{len(b - orfaos):,}",
        "orfaos": f"{len(orfaos):,}",
    })

display(pd.DataFrame(resultados))
print("\n'Órfãos' são valores presentes na segunda tabela que não existem na primeira —")
print("é o que quebraria um join e faria linhas desaparecerem sem aviso.")

## 7. Como manter

Três regras curtas:

1. **Coluna nova na extração** → acrescente a entrada no `CATALOGO` da seção 2 e rode este notebook. A seção 4 avisa se você esquecer.
2. **Nunca edite o `DICIONARIO_DE_DADOS.md`** à mão — ele é sobrescrito.
3. **Rode este notebook por último**, depois das extrações. Ele documenta o que existe no disco; fontes ainda não extraídas simplesmente não aparecem.

O `dicionario_dados.csv` serve para carregar no Autonomous Database como tabela de metadados, o que permite ao Select AI responder perguntas sobre o próprio modelo de dados — "que colunas têm valor em reais?", "onde está a população?" — sem que alguém precise manter essa descrição em outro lugar.